# Insect Trajectory Prediction — Multi-step & the KUN scheme

**TP3 (part 2) — Unified Notebook (PyTorch)**

- Binôme 1:
- Binôme 2:


`Objectives`: forecast a **full horizon** `H` of the 2-D insect trajectory in one go (**direct
multi-output**), add the **KUN (Kernel U-Net)** scheme, and **compare computation time vs accuracy**.

This is the sequel to [`03_insect_trajectory.ipynb`](03_insect_trajectory.ipynb): **same dataset,
same models**, but here every model emits the **whole horizon at once** instead of being rolled out.

1. Slice the series into `(lookback L -> horizon H)` samples.
2. Build several **direct multi-output** models: `Linear` (worked example), then `MLP` / `RNN` /
   `LSTM` / `GRU` / `Transformer` (**for you to implement**).
3. Build a simple **2-layer KUN** following the interface of [`kun_lib_v2.py`](kun_lib_v2.py). Its
   per-node **kernel** is pluggable — `linear` (worked example), then `mlp` / `lstm` / `transformer`
   (**for you to implement**). The kernel is where the Section-5 architectures plug in, *inside* the
   hierarchy.
4. Benchmark **both accuracy and cost** — test error, **training time**, and parameter count.
5. Run a **scaling experiment**: grow `L = H` from 32 to 256, **plot how the compute time grows with
   the data length** (alongside the error), and **report the accuracy as a table**.

## Notebook structure

Run **top to bottom**. **Section 1** collects the hyper-parameters. **Section 2** simulates the
insect, **Section 3** builds the standardised split and the `make_dataset` + `make_loaders` data-processing helpers (reused everywhere after), **Section 4** shows the default `(L -> H)`
windows. **Section 5** defines the **kernel protocol** -- the single building block every model is
built from -- with the `linear` kernel given (the `mlp` / `rnn` / `lstm` / `gru` / `transformer`
kernels are `TODO`) and a `DirectKernel` wrapper (given) that turns any kernel into a direct
one-call model. **Section 6** reuses *those same kernels* inside a simple 2-layer **KUN** following
`kun_lib_v2.py`: the encoder/decoder is wired up for you (`__init__`), but the **`forward` that
applies the kernels** -- the patch-split + per-patch dispatch -- is a `TODO`. **Section 7** is the
train / evaluate utilities, **Section 8** trains every enabled model and reports a benchmark of
**accuracy, training time and size**, **Section 9** plots the per-horizon error and forecast
trajectories, and **Section 10** runs the `L = H` scaling experiment comparing **error and compute
time**. **Section 11** is the conclusion.

Nothing runs end-to-end until you complete the `linear` kernel (Section 5, given) **and** the
`SimpleKUNet.forward` (Section 6); once `Linear` + `KUN-linear` work, the tables and plots update
automatically as you implement and enable more models in the `MODELS` factory.

In [ ]:
import random
import time
from math import sin, cos

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)
np.random.seed(0)
print('Using device:', device)

## 1. Parameters

All hyper-parameters in one place. `predict_len` (`H`) is now a **direct** forecast horizon.

In [ ]:
# ---- Dataset ---------------------------------------------------
max_t        = 100      # simulation horizon (time units)
delta_t      = 0.01     # time step
features_len = 2        # channels C : (x, y)

sequence_len = 32       # lookback L : length of the input window
predict_len  = 32       # horizon  H : number of steps to forecast at once

# ---- Training --------------------------------------------------
train_len    = 1600     # train time steps (chronological, comes first)
test_len     = 400      # test  time steps (held out strictly AFTER train -> no leakage)
train_prop   = 0.8      # reused by the Section-10 sweep
batch_size   = 32
epochs       = 50
lr           = 3e-4

## 2. Insect simulation

Identical to notebook 3: a smooth, quasi-periodic 2-D motion built from sines and cosines.

In [ ]:
def insect_init(s=122):
    if s > 0:
        random.seed(s)
    insect_init.params_x = [random.gauss(0., 1.) for _ in range(8)]
    insect_init.params_y = [random.gauss(0., 1.) for _ in range(8)]


def insect_move(t):
    [ax1, ax2, ax3, ax4, kx1, kx2, kx3, kx4] = insect_init.params_x
    [ay1, ay2, ay3, ay4, ky1, ky2, ky3, ky4] = insect_init.params_y

    x = ax1*sin(t*(kx1+20)) + ax2*cos(t*(kx2+10)) + ax3*sin(t*(kx3+5)) + ax4*cos(t*(kx4+5))
    y = ay1*cos(t*(ky1+20)) + ay2*sin(t*(ky2+10)) + ay3*cos(t*(ky3+5)) + ay4*sin(t*(ky4+5))
    return x, y

## 3. Build the dataset (generate -> split -> standardise)

Generate the trajectory, split it chronologically, and standardise per channel with the **train**
statistics only (no leakage).

In [ ]:
# ---- Generate the trajectory
insect_init(s=16)
dataset = [insect_move(t) for t in np.arange(0., max_t, delta_t)]


# ======================================================================
# The ONLY data-processing helpers. Every later section uses these --
# nothing else re-splits, re-standardises or re-windows.
# ======================================================================
def make_dataset(dataset, train_len, test_len, sequence_len, predict_len):
    """Chronological split + per-channel standardisation on TRAIN stats only. Returns the
    standardised (train_set, test_set): test is strictly AFTER train and the stats never see it,
    so there is no leakage."""
    assert train_len + sequence_len + predict_len  + test_len + sequence_len + predict_len <= len(dataset), 'not enough simulated points'
    data = np.array(dataset[:train_len + sequence_len + predict_len + test_len + sequence_len + predict_len], dtype='float32')
    train_set = data[:train_len + sequence_len + predict_len - 1]
    test_set  = data[train_len + sequence_len + predict_len - 1 :train_len + sequence_len + predict_len + test_len + sequence_len + predict_len - 1 - 1 ]
    mean = train_set.mean(axis=0)
    std  = train_set.std(axis=0)
    return (train_set - mean) / std, (test_set - mean) / std


def make_loaders(train_set, test_set, L, H, batch_size=batch_size):
    """Window each standardised split into a DataLoader (L -> H). Windows are built per split, so
    none straddles the train/test boundary -> no leakage. Recover the raw arrays via
    loader.dataset.tensors."""
    def windows(data):                       # (L -> H) sliding windows over one split
        X, Y = [], []
        for i in range(len(data) - L - H + 1):
            X.append(data[i:i+L])
            Y.append(data[i+L:i+L+H])
        return (torch.from_numpy(np.array(X, dtype='float32')),
                torch.from_numpy(np.array(Y, dtype='float32')))

    train_loader = DataLoader(TensorDataset(*windows(train_set)), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TensorDataset(*windows(test_set)),  batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


# ---- Build the dataset (train_set, test_set) + default loaders, reused everywhere after ----
train_set, test_set = make_dataset(dataset, train_len, test_len, sequence_len, predict_len)
train_loader, test_loader = make_loaders(train_set, test_set, sequence_len, predict_len)
print('Train shape :', train_set.shape, '  Test shape :', test_set.shape)
print('Train  loader shape :', len(train_loader.dataset), '  Test loader shape :', len(test_loader.dataset))

In [ ]:
train_loader.dataset

In [ ]:
plt.figure(figsize=(6, 5))
plt.plot(train_set[:, 0], train_set[:, 1], c='tab:blue', lw=1, alpha=0.6, label='Train')
plt.plot(test_set[:, 0],  test_set[:, 1],  c='tab:red',  lw=1, alpha=0.6, label='Test')
plt.xlabel('x (normalized)'); plt.ylabel('y (normalized)')
plt.title('Insect trajectory: train / test')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## 4. Multi-step windows (L -> H)

Each sample maps a lookback window to the **whole** horizon:
`X[i] = data[i : i+L]` (shape `L x C`) -> `Y[i] = data[i+L : i+L+H]` (shape `H x C`).

In [ ]:
# Default (L = H = sequence_len) loaders were built in Section 3 via make_loaders.
print('X_train:', tuple(train_loader.dataset.tensors[0].shape),
      ' Y_train:', tuple(train_loader.dataset.tensors[1].shape))
print('X_test :', tuple(test_loader.dataset.tensors[0].shape),
      ' Y_test :', tuple(test_loader.dataset.tensors[1].shape))
print('batches / epoch:', len(train_loader), '(train),', len(test_loader), '(test)')

## 5. The kernel protocol -- one building block for every model

Every model in this notebook -- the **direct** baselines here *and* the **KUN** hierarchy in
Section 6 -- is built from a single building block, the **kernel** (the `kun_lib_v2.py` interface):

> `Kernel(input_shape, output_shape, input_dim, output_dim)` maps one chunk
> `(*input_shape, input_dim) -> (*output_shape, output_dim)`.

A **direct multi-output** model is the simplest possible use: **one kernel call** over the whole
window -- `input_shape = (L,)` to `output_shape = (H,)` with `input_dim = output_dim = C`. Swapping
the kernel (`linear` / `mlp` / `rnn` / `lstm` / `gru` / `transformer`) gives every Section-5 model.
Section 6 reuses *the exact same kernels*, just called many times inside a U-Net hierarchy.

`linear` is the **worked example**. Implement the other kernels yourself: replace every
`# >>> YOUR CODE HERE <<<` block and delete the `raise NotImplementedError` line. The shared
`DirectKernel` wrapper (given) then turns any kernel into a direct model.

In [ ]:
from math import prod   # Python 3.8+

# ---- The kernel protocol: every model's building block (kun_lib_v2.py interface) ----
class Kernel(nn.Module):
    """Base kernel: maps (..., *input_shape, input_dim) -> (..., *output_shape, output_dim)."""
    def __init__(self, input_shape, output_shape, input_dim, output_dim, **kw):
        super().__init__()
        self.input_shape  = tuple(input_shape)
        self.output_shape = tuple(output_shape)
        self.input_dim    = input_dim
        self.output_dim   = output_dim


class LinearKernel(Kernel):
    """Linear kernel (worked example): flatten the chunk -> Linear -> reshape."""
    def __init__(self, input_shape, output_shape, input_dim, output_dim, **kw):
        super().__init__(input_shape, output_shape, input_dim, output_dim, **kw)
        self.fc = nn.Linear(prod(input_shape) * input_dim, prod(output_shape) * output_dim)
    def forward(self, x):
        x = x.reshape(-1, prod(self.input_shape) * self.input_dim)
        x = self.fc(x)
        return x.reshape(-1, *self.output_shape, self.output_dim)


# ======================================================================
# TODO (PLACEHOLDER): implement the kernels below (cf. kun_lib_v2.py).
# Same interface as LinearKernel; only the inner map changes.
# ======================================================================
class MLPKernel(Kernel):
    def __init__(self, input_shape, output_shape, input_dim, output_dim, hidden=64, **kw):
        super().__init__(input_shape, output_shape, input_dim, output_dim, **kw)
        # >>> YOUR CODE HERE <<< : Linear(in, hidden) -> GELU -> Linear(hidden, out)
        raise NotImplementedError("TODO: MLP kernel")
    def forward(self, x):
        # >>> YOUR CODE HERE <<< : reshape -> net -> reshape to (-1, *output_shape, output_dim)
        raise NotImplementedError


class RNNKernel(Kernel):
    """Run an RNN/LSTM/GRU over the chunk's token axis -> last hidden state -> project.

    The chunk is read as prod(input_shape) tokens of width input_dim. With input_shape=(L,) this is
    a direct recurrent model over the window; inside KUN it processes one patch. The subclasses
    LSTMKernel / GRUKernel only flip the `cell` attribute.
    """
    cell = 'RNN'
    def __init__(self, input_shape, output_shape, input_dim, output_dim, hidden=64, **kw):
        super().__init__(input_shape, output_shape, input_dim, output_dim, **kw)
        # >>> YOUR CODE HERE <<< : pick nn.RNN / nn.LSTM / nn.GRU from self.cell (batch_first=True),
        #     input width = input_dim, then a Linear(hidden, prod(output_shape)*output_dim) head
        raise NotImplementedError("TODO: recurrent kernel")
    def forward(self, x):
        # >>> YOUR CODE HERE <<< : reshape to (B, prod(input_shape), input_dim), run the rnn,
        #     take the LAST token's hidden state, project, reshape to (-1, *output_shape, output_dim)
        raise NotImplementedError


class LSTMKernel(RNNKernel):
    cell = 'LSTM'


class GRUKernel(RNNKernel):
    cell = 'GRU'


class TransformerKernel(Kernel):
    def __init__(self, input_shape, output_shape, input_dim, output_dim,
                 d_model=32, n_heads=4, n_layers=1, **kw):
        super().__init__(input_shape, output_shape, input_dim, output_dim, **kw)
        # >>> YOUR CODE HERE <<< : (1) project each token input_dim -> d_model (e.g. 32) so attention
        #     has enough width -- at the first level input_dim = C = 2, far too narrow on its own;
        #     (2) TransformerEncoder over the prod(input_shape) tokens; (3) flatten -> Linear to
        #     prod(output_shape)*output_dim. NB d_model must be divisible by n_heads. (No positional
        #     encoding -- the final per-position Linear already encodes token order.)
        raise NotImplementedError("TODO: Transformer kernel")
    def forward(self, x):
        # >>> YOUR CODE HERE <<<
        raise NotImplementedError


KERNEL_MAP = {'linear': LinearKernel, 'mlp': MLPKernel,
              'rnn': RNNKernel, 'lstm': LSTMKernel, 'gru': GRUKernel,
              'transformer': TransformerKernel}


class DirectKernel(nn.Module):
    """A direct multi-output model = ONE kernel call over the whole window: (L, C) -> (H, C). GIVEN.

    The kernel protocol IS the model here: input_shape=(L,), output_shape=(H,), in/out dim = C.
    Section 6 reuses the very same kernels, but calls them repeatedly inside the KUN hierarchy.
    """
    def __init__(self, kernel, L, H, C):
        super().__init__()
        self.kernel = KERNEL_MAP[kernel]((L,), (H,), C, C)
    def forward(self, x):                # x: (B, L, C) -> (B, H, C)
        return self.kernel(x)

## 6. KUN (Kernel U-Net) -- the paper's interface

We follow the interface of [`kun_lib_v2.py`](kun_lib_v2.py) (the real, N-dimensional KUN), but build a
deliberately **simple 2-layer** version for our 1-D series. It reuses the **same kernels** from Section 5.

### The one idea: split the length, then dispatch every patch to a kernel

A window arrives as **`(B, L, C)`** -- `B` sequences, length `L`, `C` channels. KUN never feeds the whole
length to one big layer. At **every level** it **chunks the length axis** into patches and runs a small,
shared **kernel** on each patch independently:

```
(B, L, C)                      one window of length L
   |   split  L = P * Lp
   v
(B, P, Lp, C)                  P patches, each of length Lp
   |   merge the batch and patch axes -- every patch is an independent kernel call
   v
(B*P, Lp, C)                   a flat batch of patches
   |   kernel:  (Lp, C_in) --> (1, C_out)
   v
(B*P, 1, C_out)
   |   reshape back
   v
(B, P, C_out)                  the P patches become the new, shorter length
```

This `(B, L, C) -> (B, P, Lp, C) -> dispatch to the kernel` split **is** the U-Net **down-sampling**: one
level turns a length-`L` sequence into a length-`P` one (`P < L`), and the kernel is what does the work.

### The interface (the Section-5 kernel protocol)

A direct model was *one* kernel call; KUN is *many*. Each patch is a kernel call with `input_shape = (Lp,)`,
`output_shape = (1,)`, and `input_dim` / `output_dim` the channel widths. A `KernelWrapper` adds the
spatial reshape and the **U-Net skip** (`encode` saves its input; the mirrored `decode` adds it back). This
is exactly how `kun_lib_v2.py` is organised -- same names, same shapes, just 1-D and 2 levels.

### The 2-layer U-Net

With `L = Lp * P` we chunk **twice** (the decoder mirrors the encoder):

```
encoder:  (L, C) --split: P patches of len Lp--> (P, hidden) --1 chunk of len P--> (1, latent)
decoder:  (1, latent) ------expand to P------> (P, hidden) ------expand to L-----> (L, C)
                            ^ skip ---------------------------------+   (matching levels added back)
```

Because this notebook keeps `H == L`, the reconstruction length equals the horizon, so `SimpleKUNet`
outputs `(B, H, C)` directly.

### Pluggable kernels

The kernels are the **same classes** from Section 5 (`linear` / `mlp` / `rnn` / `lstm` / `gru` /
`transformer`). Swapping the kernel is how KUN trades speed for expressiveness -- *the kernel is where the
Section-5 models plug in, now inside the hierarchy instead of as a single call.*

> **TODO (this section).** The `KernelWrapper` and the `SimpleKUNet.__init__` wiring are **given**;
> **you implement `SimpleKUNet.forward`** -- the part where KUN *applies the kernels*: at each level
> split the length into patches, dispatch every patch to the right `enc`/`dec` node, and reshape
> back (the U-Net skips are wired for you). Follow the reshape recipe in the docstring / comments.

In [ ]:
class KernelWrapper(nn.Module):
    """Wrap a kernel with the spatial reshape + U-Net skip (kun_lib_v2.py interface)."""
    def __init__(self, kernel_cls, input_shape, output_shape, input_dim, output_dim,
                 mode='encode', unet_skip=True, **kw):
        super().__init__()
        self.input_shape  = tuple(input_shape)
        self.output_shape = tuple(output_shape)
        self.input_dim    = input_dim
        self.output_dim   = output_dim
        self.mode = mode
        self.unet_skip = unet_skip
        self._skip_saved = None
        kcls = KERNEL_MAP[kernel_cls] if isinstance(kernel_cls, str) else kernel_cls
        self.kernel = kcls(input_shape, output_shape, input_dim, output_dim, **kw)

    def forward(self, x):
        x = x.reshape(-1, *self.input_shape, self.input_dim)
        if self.unet_skip and self.mode == 'encode':
            self._skip_saved = x                     # save activation for the decoder
        x = self.kernel(x)
        x = x.reshape(-1, *self.output_shape, self.output_dim)
        if self.unet_skip and self.mode == 'decode':
            x = x + self._skip_saved                 # add the matching encoder activation
        return x

In [ ]:
def two_factors(L):
    """Split L into two balanced factors (Lp, P) with Lp * P == L.

    Lp = patch length (the fine chunk handed to the kernel); P = number of patches.
    """
    a = int(round(L ** 0.5))
    while L % a:
        a -= 1
    return (L // a, a)                          # (Lp, P) = (patch length, number of patches)


class SimpleKUNet(nn.Module):
    """A simple 2-layer Kernel U-Net, faithful to kun_lib_v2.py. Assumes H == L.

    The __init__ (GIVEN) wires four KernelWrappers into a 2-level encoder/decoder; the `forward`
    (TODO -- for you) is where KUN actually APPLIES the kernel: at every level it splits the length
    axis into patches and dispatches each patch to a (shared) kernel.

    The single operation, repeated at every level, is a PATCH SPLIT + a per-patch kernel:

        (B, L, C) --reshape--> (B, P, Lp, C)   # split length L into P patches of length Lp
                  --merge----> (B*P, Lp, C)     # every patch is an independent kernel call
                  --kernel---> (B*P, 1,  dim)   # kernel: (Lp, C_in) -> (1, C_out)
                  --reshape--> (B, P,  dim)     # the P patches become the new, shorter length

    Two levels chain this; the decoder mirrors it with U-Net skips:
        encoder:  (L, C) --[P patches, len Lp]--> (P, hidden) --[1 chunk, len P]--> (1, latent)
        decoder:  (1, latent) ----expand P----> (P, hidden) ----expand L----> (L, C)
    """
    def __init__(self, L, H, C, kernel='linear', hidden_dim=32, latent_dim=64, unet_skip=True):
        super().__init__()
        assert H == L, 'this simple 2-layer KUN assumes H == L (true throughout this notebook)'
        Lp, P = two_factors(L)                  # L = Lp * P : patch length, number of patches
        self.Lp, self.P, self.C, self.L = Lp, P, C, L
        # ---- GIVEN: the four kernel nodes (encoder down, decoder up). Each is the Section-5
        #      kernel wrapped with the spatial reshape + U-Net skip.
        #                  kernel  input_shape output_shape in_dim      out_dim
        self.enc1 = KernelWrapper(kernel, (Lp,), (1,),  C,          hidden_dim, mode='encode', unet_skip=unet_skip)
        self.enc2 = KernelWrapper(kernel, (P,),  (1,),  hidden_dim, latent_dim, mode='encode', unet_skip=unet_skip)
        self.dec1 = KernelWrapper(kernel, (1,),  (P,),  latent_dim, hidden_dim, mode='decode', unet_skip=unet_skip)
        self.dec2 = KernelWrapper(kernel, (1,),  (Lp,), hidden_dim, C,          mode='decode', unet_skip=unet_skip)

    # ======================================================================
    # TODO (PLACEHOLDER): apply the kernels -- run the 2-level KUN forward pass.
    # Use self.enc1 / self.enc2 / self.dec1 / self.dec2 (built above) and the
    # reshape recipe in the docstring. The decoder's U-Net skips are wired for
    # you. Delete the `raise NotImplementedError` once done.
    # ======================================================================
    def forward(self, x):                        # x: (B, L, C)  ->  (B, H=L, C)
        B, Lp, P, C = x.shape[0], self.Lp, self.P, self.C

        # ---- ENCODER ----
        # Level 1: split length L into P patches of length Lp, then DISPATCH every patch to enc1.
        #   >>> YOUR CODE HERE <<<
        #   patches : reshape x (B, L, C) -> (B, P, Lp, C) -> (B*P, Lp, C)
        #   e1      : self.enc1(patches) then reshape back to (B, P, hidden)
        # Level 2: the whole length-P sequence is one chunk -> latent token.
        #   z       : self.enc2(e1)                                  # (B, P, hidden) -> (B, 1, latent)
        raise NotImplementedError("TODO: KUN encoder -- patch split + kernel dispatch")

        # ---- DECODER (mirror) with U-Net skips: dec level i <- enc level (n-1-i) ----
        # (GIVEN) hand each decoder node the matching encoder activation:
        self.dec1._skip_saved = self.enc2._skip_saved
        self.dec2._skip_saved = self.enc1._skip_saved
        #   >>> YOUR CODE HERE <<<
        #   d1 : self.dec1(z) then reshape to (B*P, 1, hidden)       # (B, 1, latent) -> (B*P, 1, hidden)
        #   d2 : self.dec2(d1)                                       # (B*P, 1, hidden) -> (B*P, Lp, C)
        #   return d2 reshaped to (B, self.L, C)                     # (B*P, Lp, C) -> (B, L=H, C)


# Factory: name -> builder(L, H, C). Direct models = ONE kernel call; KUN = the same kernels,
# stacked into a 2-level hierarchy. Only Linear + KUN-linear are active by default.
# TODO (PLACEHOLDER): uncomment each line once that kernel is implemented.
MODELS = {
    'Linear':          lambda L, H, C: DirectKernel('linear',      L, H, C),
    # 'MLP':             lambda L, H, C: DirectKernel('mlp',         L, H, C),          # TODO
    # 'RNN':             lambda L, H, C: DirectKernel('rnn',         L, H, C),          # TODO
    # 'LSTM':            lambda L, H, C: DirectKernel('lstm',        L, H, C),          # TODO
    # 'GRU':             lambda L, H, C: DirectKernel('gru',         L, H, C),          # TODO
    # 'Transformer':     lambda L, H, C: DirectKernel('transformer', L, H, C),          # TODO
    'KUN-linear':      lambda L, H, C: SimpleKUNet(L, H, C, kernel='linear'),
    # 'KUN-mlp':         lambda L, H, C: SimpleKUNet(L, H, C, kernel='mlp'),            # TODO
    # 'KUN-lstm':        lambda L, H, C: SimpleKUNet(L, H, C, kernel='lstm'),           # TODO
    # 'KUN-transformer': lambda L, H, C: SimpleKUNet(L, H, C, kernel='transformer'),    # TODO
}

## 7. Experiment: train & evaluate

Reusable utilities (same shape as notebook 3): `evaluate` returns the multi-step `(MSE, MAE)`;
`train_model` trains with Adam + MSE and records the per-epoch train / val loss. `direct_predict`
and `per_horizon_rmse` are used later for the horizon analysis.

In [ ]:
def evaluate(model, loader):
    """Return multi-step (MSE, MAE) over a data loader."""
    model.eval()
    mse_sum, mae_sum, n = 0.0, 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)                                  # (B, H, C)
            mse_sum += nn.functional.mse_loss(pred, yb, reduction='sum').item()
            mae_sum += nn.functional.l1_loss(pred, yb, reduction='sum').item()
            n += yb.numel()
    return mse_sum / n, mae_sum / n


def train_model(model, train_loader, test_loader, epochs=epochs, lr=lr, verbose=False):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    history = {'train': [], 'val': []}
    for ep in range(epochs):
        model.train()
        run = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            run += loss.item() * len(xb)
        history['train'].append(run / len(train_loader.dataset))
        history['val'].append(evaluate(model, test_loader)[0])
        if verbose:
            print(f'  epoch {ep+1:2d}/{epochs}  val {history["val"][-1]:.4f}')
    return model, history


def count_params(model):
    return sum(p.numel() for p in model.parameters())


@torch.no_grad()
def direct_predict(model, X):
    model.eval()
    return model(torch.from_numpy(X).to(device)).cpu().numpy()


def per_horizon_rmse(pred, true):
    """(N, H, C) -> RMSE at each horizon step, shape (H,)."""
    return np.sqrt(((pred - true) ** 2).mean(axis=(0, 2)))

## 8. Compare the models — accuracy, time & size

Train every model in `MODELS`, timing each run, then summarise accuracy **and** cost side by side
(test MSE / MAE, parameter count, wall-clock training time). By default only `Linear` and
`KUN-linear` run; the table and plots grow automatically as you enable more.

In [ ]:
results = {}
for name, build in MODELS.items():
    print(f'Training {name} ...')
    torch.manual_seed(0)
    model = build(sequence_len, predict_len, features_len)
    n_params = count_params(model)
    t0 = time.time()
    model, history = train_model(model, train_loader, test_loader)
    train_time = time.time() - t0
    mse, mae = evaluate(model, test_loader)
    results[name] = {'model': model, 'history': history, 'mse': mse, 'mae': mae,
                     'params': n_params, 'train_time': train_time}

# ---- Benchmark table: accuracy + cost ----
benchmark = pd.DataFrame(
    [{'Model': n,
      f'{predict_len}-step MSE': r['mse'],
      f'{predict_len}-step MAE': r['mae'],
      'params': r['params'],
      'train time (s)': round(r['train_time'], 2)} for n, r in results.items()]
).set_index('Model').sort_values(f'{predict_len}-step MSE')
benchmark

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

for name, r in results.items():
    ax1.plot(r['history']['val'], label=name)
ax1.set_xlabel('epoch'); ax1.set_ylabel('validation MSE')
ax1.set_title('Validation loss curves'); ax1.legend(); ax1.grid(True)

names = list(results.keys())
ax2.bar(names, [results[n]['mse'] for n in names], color='tab:blue')
ax2.set_ylabel('test MSE'); ax2.set_title(f'{predict_len}-step test MSE')
ax2.tick_params(axis='x', rotation=30)

ax3.bar(names, [results[n]['train_time'] for n in names], color='tab:orange')
ax3.set_ylabel('training time (s)'); ax3.set_title('Training time')
ax3.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 9. Multi-step forecast: per-horizon error & trajectories

The **per-horizon curve** shows how the error grows with how far ahead we predict. Then we plot a
few predicted vs. true trajectories for the best model.

In [ ]:
# ---- per-horizon RMSE + add the mean to the benchmark ----
Xte, Yte = (t.numpy() for t in test_loader.dataset.tensors)   # raw test arrays from the loader
for name, r in results.items():
    r['per_horizon'] = per_horizon_rmse(direct_predict(r['model'], Xte), Yte)
benchmark['mean RMSE'] = [results[n]['per_horizon'].mean() for n in benchmark.index]
benchmark = benchmark.sort_values('mean RMSE')
best_name = benchmark.index[0]
print('Best model:', best_name)

plt.figure(figsize=(9, 5))
steps = np.arange(1, predict_len + 1)
for name, r in sorted(results.items(), key=lambda kv: kv[1]['per_horizon'].mean()):
    style = '-o' if name.startswith('KUN') else '-'
    plt.plot(steps, r['per_horizon'], style, ms=3, label=name)
plt.xlabel('forecast step h'); plt.ylabel('RMSE')
plt.title('Error vs. forecast horizon (lower & flatter is better)')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
benchmark

In [ ]:
# ---- trajectories of the best model ----
best_model = results[best_name]['model']
starts = np.linspace(0, len(Xte) - 1, 3, dtype=int)

fig, axes = plt.subplots(1, len(starts), figsize=(15, 4.5))
for ax, s in zip(axes, starts):
    window = Xte[s]
    true   = Yte[s]
    pred   = direct_predict(best_model, Xte[s:s+1])[0]
    ax.plot(window[:, 0], window[:, 1], 'o-', c='tab:blue',  ms=3, label='input')
    ax.plot(true[:, 0],   true[:, 1],   'o-', c='tab:green', ms=3, label='true')
    ax.plot(pred[:, 0],   pred[:, 1],   'x--', c='tab:red',  ms=4, label='predicted')
    ax.set_title(f'start={s}'); ax.grid(True)
axes[0].legend()
fig.suptitle(f'{best_name}: direct {predict_len}-step forecast')
plt.tight_layout(); plt.show()

## 10. Scaling experiment: `L = H` from 32 to 256 -- accuracy *and* compute time

How do **error** and **cost** change as we grow the lookback and horizon together? For
`L = H in {32, 64, 128, 256}` we reuse the **exact same standardised 1600 / 400 split as Section 3**
(`train_set` / `test_set`, train/test gap built in) and re-window it via `make_loaders` for each
`(L, H)` -- so the `L = 32` point reproduces the Section-8 benchmark and there is no train/test
leakage. Any length whose 400-point test split is too short to form one `(L -> H)` window is skipped
and reported, so with `test_len = 400` the sweep runs `L = 32, 64, 128` (`L = 256` would need 512
test points). We record both the mean RMSE **and** the training time.

> **Required deliverables (this is the graded part):**
> 1. **A plot of compute time vs. data length** `L` -- show how training time grows as `L` increases,
>    shown next to the accuracy-vs-length plot.
> 2. **A table of the accuracy** (mean RMSE) of every model at each length `L`.

> Cost grows with `L`; with only `Linear` + `KUN-linear` enabled it runs in a few minutes on CPU.
> Reduce `sweep_epochs` if needed. On a GPU you can enable every model.

In [ ]:
sweep_Ls     = [32, 64, 128, 256]
sweep_epochs = epochs        # same as Section 8 -> the L = 32 point is directly comparable

sweep_rmse = {name: [] for name in MODELS}
sweep_time = {name: [] for name in MODELS}
for L in sweep_Ls:
    H = L
    # reuse the SAME data processing as Section 3 -> just a new (L, H)
    train_set, test_set = make_dataset(dataset, train_len, test_len, L, H)
    train_loader, test_loader = make_loaders(train_set, test_set, L, H)
    print('Train shape :', train_set.shape, '  Test shape :', test_set.shape)
    print('Train  loader shape :', len(train_loader.dataset), '  Test loader shape :', len(test_loader.dataset))
    Xb, Yb = (t.numpy() for t in test_loader.dataset.tensors)
    for name, build in MODELS.items():
        torch.manual_seed(0)
        t0 = time.time()
        model, _ = train_model(build(L, H, features_len), train_loader, test_loader, epochs=sweep_epochs)
        dt = time.time() - t0
        rmse = per_horizon_rmse(direct_predict(model, Xb), Yb).mean()
        sweep_rmse[name].append(rmse)
        sweep_time[name].append(dt)
        print(f'L=H={L:3d}  {name:12s}  mean RMSE {rmse:.4f}   train {dt:6.1f}s')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for name in MODELS:
    ax1.plot(sweep_Ls, sweep_rmse[name], '-o', label=name)
ax1.set_xlabel('L = H  (lookback = horizon)'); ax1.set_ylabel('mean RMSE')
ax1.set_xscale('log', base=2); ax1.set_xticks(sweep_Ls); ax1.set_xticklabels(sweep_Ls)
ax1.set_title('Accuracy vs. length'); ax1.legend(); ax1.grid(True)

for name in MODELS:
    ax2.plot(sweep_Ls, sweep_time[name], '-o', label=name)
ax2.set_xlabel('L = H  (lookback = horizon)'); ax2.set_ylabel('training time (s)')
ax2.set_xscale('log', base=2); ax2.set_xticks(sweep_Ls); ax2.set_xticklabels(sweep_Ls)
ax2.set_title('Compute time vs. length'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# ---- Required deliverable: accuracy + compute-time TABLES vs data length ----
acc_table = pd.DataFrame({n: [round(v, 4) for v in sweep_rmse[n]] for n in MODELS}, index=sweep_Ls)
acc_table.index.name = 'L = H'
time_table = pd.DataFrame({n: [round(v, 2) for v in sweep_time[n]] for n in MODELS}, index=sweep_Ls)
time_table.index.name = 'L = H'

print('Mean RMSE vs. data length (accuracy table):')
display(acc_table)
print('\nTraining time in seconds vs. data length (compute-cost table):')
display(time_table)

## 11. Conclusion

_Fill in from your own run._

**Models implemented:** Linear (given), KUN-linear (given), ... _(list the ones you added)_

**Accuracy vs. cost** (Section 8):
- Best model by accuracy (lowest mean RMSE): ...
- Cheapest model (fewest params / shortest training time): ...
- Is the most accurate model also the most expensive? Where is the sweet spot? ...

**Per-horizon error** (Section 9):
- Does KUN's per-horizon curve stay flatter than the others as `h` grows? ...
- KUN: did swapping the `linear` kernel for `mlp` / `lstm` / `transformer` help? Was it worth the cost? ...

**Scaling experiment** (Section 10):
- How does the **error** change as `L = H` grows from 32 to 256? ...
- How does the **training time** grow with `L`? Which model scales better? ...
- Putting the two plots together: which model gives the best accuracy *per second* at long horizons? ...

**Takeaway:** **direct multi-output** forecasts the whole horizon at once, and **KUN** adds a
multi-scale hierarchy with a pluggable kernel (interface from `kun_lib_v2.py`) — letting you trade
compute for accuracy by swapping the kernel. See [the real KUN](https://jiangyou2025.github.io/kun/zh/kun/).
